In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
DROP TABLE IF EXISTS default.students;
DROP TABLE IF EXISTS default.students_update;
DROP TABLE IF EXISTS default.enrollments;
DROP TABLE IF EXISTS default.courses;

# Datei einlesen und als Tabelle abspeichern

In [0]:
%python
dataset_school = "/Volumes/workspace/default/volume"

all_files = dbutils.fs.ls(dataset_school)
json_files = [f for f in all_files if f.name.endswith(".json")]

display(json_files)

In [0]:
# JSON lesen:
students_df = spark.read.json("/Volumes/workspace/default/volume/students.json")
students_df.createOrReplaceTempView("students")

studentsupdate_df = spark.read.json("/Volumes/workspace/default/volume/students_update.json")
studentsupdate_df.createOrReplaceTempView("students_update")


courses_df = spark.read.json("/Volumes/workspace/default/volume/courses.json")
courses_df.createOrReplaceTempView("courses")

enrollments_df = spark.read.json("/Volumes/workspace/default/volume/enrollments.json")
enrollments_df.createOrReplaceTempView("enrollments")

In [0]:
# 1) Students speichern
(students_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") # Erlaubt Schema-Anpassungen
    .saveAsTable("workspace.default.students"))

# 2) Students Update speichern
(studentsupdate_df.write 
    .format("delta") 
    .mode("overwrite") 
    .saveAsTable("workspace.default.studentsupdate"))

# 3) Courses speichern
(courses_df.write 
    .format("delta") 
    .mode("overwrite") 
    .saveAsTable("workspace.default.courses"))

# 4) Enrollments speichern (Die Verknüpfungstabelle)
(enrollments_df.write 
    .format("delta") 
    .mode("overwrite") 
    .saveAsTable("workspace.default.enrollments"))

In [0]:
%sql
-- Daten lesen
SELECT * FROM default.students;

In [0]:
%sql
-- Daten lesen
SELECT * FROM default.courses;

In [0]:
%sql
-- Daten lesen
SELECT * FROM default.enrollments;

# Join-Operationen in Spark SQL

In [0]:
%sql
-- Array auf einzelne Zeilen aufbrechen und joinen
CREATE OR REPLACE VIEW enrollments_enriched AS
SELECT 
    e.enroll_id,
    e.student_id,
    e.timestamp,
    e.total,
    c.course_id,
    c.title,
    c.instructor
FROM (
    SELECT 
        enroll_id,
        student_id,
        timestamp,
        total,
        explode(courses) AS course_id -- Wandelt ein Array (z. B. [102, 103]) in einzelne Zeilen um; nutzt die Built-in Function "explode".
    FROM default.enrollments
) e
INNER JOIN default.courses c
ON e.course_id = c.course_id;

-- Ergebnis anzeigen
SELECT * FROM enrollments_enriched;

Set-Operationen in Spark SQL

In [0]:
%sql
SELECT * FROM default.studentsupdate

In [0]:
%sql
SELECT * FROM students
UNION
SELECT *FROM studentsupdate

# Higher-Order Funktionen

In [0]:
%sql
--Ausgangslage
SELECT * from default.enrollments

In [0]:
%sql
-- Beispiel FILTER: nur bestimmte Kurs-IDs behalten (z. B. 103)
SELECT
  enroll_id,
  courses,
  FILTER(courses,
         c -> c = 103) AS only_103 -- FILTER(array, x -> predicate) wählt Elemente aus einem Array.
FROM enrollments
WHERE size(FILTER(courses, c -> c = 103)) > 0; -- WHERE ist optional um leere Arrays austzuschliessen

In [0]:
%sql
-- Beispiel TRANSFORM: Gesamtpreis pro Enrollment auf Kurse verteilen
-- Performance: TRANSFORM bleibt "In-Place" (wie eine "For-Each"-Schleife innerhalb einer Tabellenzelle). 
-- Es verändert die Daten direkt im Speicher der Zelle, ohne die Tabelle physisch umzustrukturieren.
SELECT
  enroll_id,
  courses,
  TRANSFORM( -- TRANSFORM(array, x -> expr) bildet jedes Element auf einen neuen Wert/STRUCT ab.
    courses,
    c -> named_struct(
           'course_id', c,
           'allocated_total', ROUND(total / size(courses), 2)
         )
  ) AS per_course_allocation
FROM enrollments;

# SQL user-defined functions (UDFs)


In [0]:
%sql
-- Beispiel 1
-- GPA (0–6.0) → Prozent (0–100)
CREATE OR REPLACE FUNCTION gpa_to_percentage(g DOUBLE)
RETURNS INT
RETURN CAST(ROUND(g * 16.66666667) AS INT); -- Höchster GPA ist 6.0. 1/6 = 16.6666667

-- Nutzung
SELECT student_id, gpa, gpa_to_percentage(gpa) AS pct
FROM default.students
UNION -- wir nehmen auch die neuen Studenten gleich mit dazu
SELECT student_id, gpa, gpa_to_percentage(gpa) AS pct 
FROM default.studentsupdate;

In [0]:
%sql
-- Beispiel 2
CREATE OR REPLACE FUNCTION get_letter_grade(g DOUBLE)
RETURNS STRING
RETURN CASE
         WHEN g >= 5.5  THEN 'A'
         WHEN g >= 3.9 THEN 'B'
         WHEN g >= 3.42 THEN 'C'
         ELSE 'F'
       END;

SELECT student_id, gpa, get_letter_grade(gpa) AS grade
FROM default.students -- Tabelle A students
UNION
SELECT student_id, gpa, get_letter_grade(gpa) AS grade 
FROM default.studentsupdate -- Tabelle B studentsupdate
;

In [0]:
%sql
DESCRIBE FUNCTION gpa_to_percentage;

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED get_letter_grade;

# Aufräumen

In [0]:
%sql
DROP FUNCTION IF EXISTS gpa_to_percentage;
DROP FUNCTION IF EXISTS get_letter_grade;

In [0]:
%sql
DROP TABLE IF EXISTS default.courses;
DROP TABLE IF EXISTS default.enrollments;
DROP TABLE IF EXISTS default.students;
DROP TABLE IF EXISTS default.studentsupdate;

In [0]:
%sql
DROP VIEW IF EXISTS default.enrollments_enriched;